# Fake News Detector: Can LLMs Spot Misinformation?

A full pipeline that:
1. Generates a labeled dataset of real vs fake news headlines
2. Tests **3 detection strategies** — zero-shot, few-shot, chain-of-thought
3. Evaluates with accuracy, precision, recall, F1
4. Builds a **Misinformation Explainer** that classifies AND explains WHY
5. Exports QLoRA-ready training data to fine-tune an open-source model

**Week 7 Community Contribution by Mirack**

Skills: Dataset curation, multi-strategy evaluation, classification metrics, fine-tuning pipeline, HuggingFace tokenizer.

In [ ]:
import os
import json
import re
import random
from openai import OpenAI
from transformers import AutoTokenizer

In [ ]:
client = OpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL", "https://openrouter.ai/api/v1"),
    api_key=os.environ.get("OPENAI_API_KEY")
)

MODEL = "gpt-4.1-nano"
tokenizer = AutoTokenizer.from_pretrained("gpt2")

## Step 1: Generate a Labeled Fake vs Real News Dataset
We generate realistic headlines across topics — half real, half fabricated with subtle misinformation.

In [ ]:
def generate_news_samples(topic, count=6):
    """Generate a mix of real-sounding and fake news headlines with articles."""
    prompt = f"""Generate exactly {count} news article samples about: {topic}
Half should be REAL (factually accurate, plausible) and half FAKE (containing subtle misinformation, exaggerated claims, or fabricated statistics).

Make the fake ones tricky — they should sound believable at first glance.

Respond ONLY with a JSON array:
[
  {{
    "headline": "...",
    "snippet": "2-3 sentence article excerpt",
    "topic": "{topic}",
    "label": "real" or "fake",
    "fake_reason": "why it's fake (empty string if real)"
  }}
]"""
    response = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}]
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"^```json\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    return json.loads(raw)

In [ ]:
topics = [
    "Climate Change", "Artificial Intelligence", "Global Economy",
    "Space Exploration", "Public Health", "Cybersecurity",
    "Renewable Energy", "Education Reform"
]

all_articles = []
for topic in topics:
    print(f"Generating {topic}...")
    articles = generate_news_samples(topic, count=6)
    all_articles.extend(articles)

random.seed(42)
random.shuffle(all_articles)

real_count = sum(1 for a in all_articles if a['label'] == 'real')
fake_count = sum(1 for a in all_articles if a['label'] == 'fake')
print(f"\nDataset: {len(all_articles)} articles | {real_count} real | {fake_count} fake")

In [ ]:
# Preview some samples
for a in all_articles[:4]:
    tag = "REAL" if a['label'] == 'real' else "FAKE"
    print(f"[{tag}] {a['headline']}")
    print(f"  {a['snippet'][:120]}...")
    if a['label'] == 'fake':
        print(f"  Why fake: {a['fake_reason']}")
    print()

In [ ]:
# Split into few-shot examples and test set
few_shot_pool = all_articles[:8]
test_set = all_articles[8:]
print(f"Few-shot pool: {len(few_shot_pool)} | Test set: {len(test_set)}")

## Step 2: Three Detection Strategies

In [ ]:
def classify_zero_shot(article):
    """Simple: just ask if it's real or fake."""
    prompt = f"""Is this news article real or fake? Respond with ONLY the word 'real' or 'fake'.

Headline: {article['headline']}
Article: {article['snippet']}"""
    response = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}]
    )
    result = response.choices[0].message.content.strip().lower()
    return "fake" if "fake" in result else "real"

In [ ]:
def classify_few_shot(article, examples):
    """Provide labeled examples so the model learns the pattern."""
    example_text = "\n".join(
        [f"Headline: {e['headline']}\nArticle: {e['snippet']}\nVerdict: {e['label']}" for e in examples[:5]]
    )
    prompt = f"""Here are some news articles with their labels:

{example_text}

Now classify this article. Respond with ONLY 'real' or 'fake'.

Headline: {article['headline']}
Article: {article['snippet']}
Verdict:"""
    response = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}]
    )
    result = response.choices[0].message.content.strip().lower()
    return "fake" if "fake" in result else "real"

In [ ]:
def classify_chain_of_thought(article):
    """Reason through red flags before classifying."""
    prompt = f"""You are a fact-checker. Analyze this news article step by step:

1. Check the headline — is it sensationalist or measured?
2. Look for specific claims — are statistics cited? Do numbers seem plausible?
3. Check for emotional manipulation — does it use fear, outrage, or too-good-to-be-true framing?
4. Assess source quality — does the writing feel professional?
5. Give your final verdict

Headline: {article['headline']}
Article: {article['snippet']}

After your analysis, end with EXACTLY: VERDICT: real  OR  VERDICT: fake"""
    response = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}]
    )
    text = response.choices[0].message.content.strip().lower()
    match = re.search(r"verdict:\s*(real|fake)", text)
    if match:
        return match.group(1)
    return "fake" if "fake" in text.split("\n")[-1] else "real"

In [ ]:
# Run all 3 strategies
predictions = []

for i, article in enumerate(test_set):
    print(f"[{i+1}/{len(test_set)}] {article['headline'][:50]}...")
    
    p_zero = classify_zero_shot(article)
    p_few = classify_few_shot(article, few_shot_pool)
    p_cot = classify_chain_of_thought(article)
    
    predictions.append({
        "headline": article['headline'],
        "actual": article['label'],
        "zero_shot": p_zero,
        "few_shot": p_few,
        "chain_of_thought": p_cot
    })

print(f"\nDone. {len(predictions)} articles classified.")

## Step 3: Evaluation — Accuracy, Precision, Recall, F1

In [ ]:
def evaluate_classifier(predictions, strategy_key):
    """Calculate classification metrics."""
    tp = sum(1 for p in predictions if p['actual'] == 'fake' and p[strategy_key] == 'fake')
    tn = sum(1 for p in predictions if p['actual'] == 'real' and p[strategy_key] == 'real')
    fp = sum(1 for p in predictions if p['actual'] == 'real' and p[strategy_key] == 'fake')
    fn = sum(1 for p in predictions if p['actual'] == 'fake' and p[strategy_key] == 'real')
    
    accuracy = (tp + tn) / len(predictions) * 100
    precision = tp / (tp + fp) * 100 if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) * 100 if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1,
            "tp": tp, "tn": tn, "fp": fp, "fn": fn}

In [ ]:
strategies = {
    "Zero-Shot": "zero_shot",
    "Few-Shot (5 examples)": "few_shot",
    "Chain-of-Thought": "chain_of_thought"
}

print(f"{'Strategy':<25} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("=" * 67)

best_strategy = None
best_f1 = 0

for name, key in strategies.items():
    m = evaluate_classifier(predictions, key)
    if m['f1'] > best_f1:
        best_f1 = m['f1']
        best_strategy = name
    print(f"{name:<25} {m['accuracy']:>9.1f}% {m['precision']:>9.1f}% {m['recall']:>9.1f}% {m['f1']:>9.1f}%")

print(f"\nBEST STRATEGY: {best_strategy} (F1: {best_f1:.1f}%)")

In [ ]:
# Confusion matrix for best strategy
best_key = strategies[best_strategy]
m = evaluate_classifier(predictions, best_key)

print(f"Confusion Matrix — {best_strategy}:\n")
print(f"                 Predicted REAL    Predicted FAKE")
print(f"  Actual REAL       {m['tn']:>4}              {m['fp']:>4}")
print(f"  Actual FAKE       {m['fn']:>4}              {m['tp']:>4}")
print(f"\nMissed fakes (False Negatives): {m['fn']} — these slipped through")
print(f"False alarms (False Positives): {m['fp']} — real articles flagged as fake")

## Step 4: Misinformation Explainer
Don't just classify — explain WHY the article is suspicious with streaming output.

In [ ]:
def explain_article(article):
    """Classify and explain with detailed reasoning via streaming."""
    prompt = f"""You are an expert fact-checker. Analyze this article and provide:

1. VERDICT: real or fake
2. CONFIDENCE: high, medium, or low
3. RED FLAGS: list any suspicious elements (empty if real)
4. REASONING: 2-3 sentences explaining your judgment

Headline: {article['headline']}
Article: {article['snippet']}

Format your response clearly with those 4 sections."""

    stream = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        stream=True
    )
    result = ""
    for chunk in stream:
        if not chunk.choices:
            continue
        content = chunk.choices[0].delta.content
        if content:
            print(content, end="")
            result += content
    return result

In [ ]:
# Run the explainer on 3 articles — mix of real and fake
sample_articles = [a for a in test_set if a['label'] == 'fake'][:2] + [a for a in test_set if a['label'] == 'real'][:1]

for article in sample_articles:
    print(f"\n{'='*60}")
    print(f"HEADLINE: {article['headline']}")
    print(f"ACTUAL LABEL: {article['label'].upper()}")
    print(f"{'='*60}\n")
    explain_article(article)
    print("\n")

## Step 5: Export QLoRA-Ready Fine-Tuning Data

In [ ]:
training_data = []
for a in all_articles:
    explanation = a.get('fake_reason', '') if a['label'] == 'fake' else 'This article appears factually accurate and uses measured language with plausible claims.'
    entry = {
        "messages": [
            {"role": "user", "content": f"Is this news real or fake? Explain your reasoning.\n\nHeadline: {a['headline']}\nArticle: {a['snippet']}"},
            {"role": "assistant", "content": f"Verdict: {a['label']}\nReasoning: {explanation}"}
        ]
    }
    training_data.append(entry)

# Token analysis
all_texts = [f"{t['messages'][0]['content']} {t['messages'][1]['content']}" for t in training_data]
token_lengths = [len(tokenizer.encode(t)) for t in all_texts]

print(f"Training samples: {len(training_data)}")
print(f"Token stats: min={min(token_lengths)}, max={max(token_lengths)}, avg={sum(token_lengths)/len(token_lengths):.0f}")
print(f"Total tokens: {sum(token_lengths):,}")
print(f"Recommended max_seq_length: {max(token_lengths) + 20}")
print(f"\nSample:")
print(json.dumps(training_data[0], indent=2))

## Key Findings

- **Chain-of-thought** generally catches more subtle misinformation by reasoning through red flags
- **Few-shot** helps calibrate what "fake" looks like but can overfit to example patterns
- **Zero-shot** is fast but misses nuanced fabrications
- The Misinformation Explainer adds real value — users need to understand WHY, not just the label
- A fine-tuned open-source model trained on this data could become a specialized fact-checking assistant